In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
import numpy as np

In [ ]:
data = pd.read_csv('/content/drive/MyDrive/merged_file.csv')
data

,date,open,high,low,close,volume,Ticker
0,2015-02-02 09:15:00,646.90,662.00,646.90,662.00,271571,ADANIENT
1,2015-02-02 09:16:00,661.70,662.30,653.60,653.90,193089,ADANIENT
2,2015-02-02 09:17:00,654.20,658.00,651.05,657.95,89069,ADANIENT
3,2015-02-02 09:18:00,657.00,657.50,655.40,656.80,68028,ADANIENT
4,2015-02-02 09:19:00,656.95,656.95,647.00,647.00,105605,ADANIENT
...,...,...,...,...,...,...,...
42857688,2024-11-08 15:25:00,568.60,568.75,568.40,568.60,27463,WIPRO
42857689,2024-11-08 15:26:00,568.60,568.60,568.00,568.10,42314,WIPRO
42857690,2024-11-08 15:27:00,568.10,568.85,568.05,568.45,53661,WIPRO
42857691,2024-11-08 15:28:00,568.45,568.45,567.75,567.80,33238,WIPRO


In [ ]:
#data = data[data['Ticker'] == "ADANIENT"]
data

,date,open,high,low,close,volume,Ticker
0,2015-02-02 09:15:00,646.90,662.00,646.90,662.00,271571,ADANIENT
1,2015-02-02 09:16:00,661.70,662.30,653.60,653.90,193089,ADANIENT
2,2015-02-02 09:17:00,654.20,658.00,651.05,657.95,89069,ADANIENT
3,2015-02-02 09:18:00,657.00,657.50,655.40,656.80,68028,ADANIENT
4,2015-02-02 09:19:00,656.95,656.95,647.00,647.00,105605,ADANIENT
...,...,...,...,...,...,...,...
42857688,2024-11-08 15:25:00,568.60,568.75,568.40,568.60,27463,WIPRO
42857689,2024-11-08 15:26:00,568.60,568.60,568.00,568.10,42314,WIPRO
42857690,2024-11-08 15:27:00,568.10,568.85,568.05,568.45,53661,WIPRO
42857691,2024-11-08 15:28:00,568.45,568.45,567.75,567.80,33238,WIPRO


In [ ]:
import numpy as np
import pandas as pd

# ------------------------- ADX Calculation -------------------------
def caladx(high, low, close, period=14):
    """Computes ADX using NumPy."""
    tr = np.maximum(high - low, np.maximum(np.abs(high - np.roll(close, 1)), np.abs(low - np.roll(close, 1))))
    dm_plus = np.where((high - np.roll(high, 1)) > (np.roll(low, 1) - low), high - np.roll(high, 1), 0)
    dm_minus = np.where((np.roll(low, 1) - low) > (high - np.roll(high, 1)), np.roll(low, 1) - low, 0)

    tr_smooth = np.convolve(tr, np.ones(period) / period, mode='same')
    dm_plus_smooth = np.convolve(dm_plus, np.ones(period) / period, mode='same')
    dm_minus_smooth = np.convolve(dm_minus, np.ones(period) / period, mode='same')

    dx = 100 * np.abs((dm_plus_smooth - dm_minus_smooth) / (dm_plus_smooth + dm_minus_smooth + 1e-9))
    adx = np.convolve(dx, np.ones(period) / period, mode='same')
    return adx

# ------------------------- MFI Calculation -------------------------
def calmfi(high, low, close, volume, period=14):
    """Computes MFI using NumPy."""
    typical_price = (high + low + close) / 3
    money_flow = typical_price * volume

    positive_flow = np.where(typical_price > np.roll(typical_price, 1), money_flow, 0)
    negative_flow = np.where(typical_price < np.roll(typical_price, 1), money_flow, 0)

    positive_mf = np.convolve(positive_flow, np.ones(period), mode='same')
    negative_mf = np.convolve(negative_flow, np.ones(period), mode='same')

    mfi = 100 - (100 / (1 + (positive_mf / (negative_mf + 1e-9))))
    return mfi

# ------------------------- VWAP Calculation -------------------------
def calvwap(data):
    """Computes VWAP using NumPy, grouped by Ticker and Date."""
    data = data.sort_values(by=['Ticker', 'date'])
    data['typical_price'] = (data['high'] + data['low'] + data['close']) / 3
    data['price_volume'] = data['typical_price'] * data['volume']

    # VWAP is computed per Ticker & Date
    data['VWAP'] = (data.groupby(['Ticker', 'date'])['price_volume'].cumsum() /
                    data.groupby(['Ticker', 'date'])['volume'].cumsum())
    return data

# ------------------------- ATR Calculation -------------------------
def calatr(high, low, close, period=14):
    """Computes ATR using NumPy."""
    tr = np.maximum(high - low, np.maximum(np.abs(high - np.roll(close, 1)), np.abs(low - np.roll(close, 1))))
    atr = np.convolve(tr, np.ones(period) / period, mode='same')
    return atr

# ------------------------- Applying Indicators Efficiently -------------------------
N = 14  # Indicator Period

def apply_indicators(group):
    """Applies ATR, ADX, and MFI to each Ticker group efficiently."""
    high, low, close, volume = group['high'].values, group['low'].values, group['close'].values, group['volume'].values
    group['ATR'] = calatr(high, low, close, N)
    group['ADX'] = caladx(high, low, close, N)
    group['MFI'] = calmfi(high, low, close, volume, N)
    return group

# Assuming 'data' is a DataFrame with columns: 'Ticker', 'date', 'high', 'low', 'close', 'volume'
# Sort the data by Ticker and date before applying indicators
data = data.sort_values(by=['Ticker', 'date'])

# Apply ATR, ADX, and MFI by Ticker
data = data.groupby('Ticker', group_keys=False).apply(apply_indicators)

# Apply VWAP by Ticker & Date
data = calvwap(data)

# Display the final DataFrame
print(data)

<ipython-input-5-3da8ff3f31d2>:69: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  data = data.groupby('Ticker', group_keys=False).apply(apply_indicators)


                         date    open    high     low   close  volume  \
0         2015-02-02 09:15:00  646.90  662.00  646.90  662.00  271571   
1         2015-02-02 09:16:00  661.70  662.30  653.60  653.90  193089   
2         2015-02-02 09:17:00  654.20  658.00  651.05  657.95   89069   
3         2015-02-02 09:18:00  657.00  657.50  655.40  656.80   68028   
4         2015-02-02 09:19:00  656.95  656.95  647.00  647.00  105605   
...                       ...     ...     ...     ...     ...     ...   
42857688  2024-11-08 15:25:00  568.60  568.75  568.40  568.60   27463   
42857689  2024-11-08 15:26:00  568.60  568.60  568.00  568.10   42314   
42857690  2024-11-08 15:27:00  568.10  568.85  568.05  568.45   53661   
42857691  2024-11-08 15:28:00  568.45  568.45  567.75  567.80   33238   
42857692  2024-11-08 15:29:00  567.95  568.80  567.70  568.25   17105   

            Ticker         ATR        ADX        MFI  typical_price  \
0         ADANIENT  165.121429  49.967240   7.745609

In [ ]:
import pandas as pd
import numpy as np
from tqdm import tqdm
from joblib import Parallel, delayed

# Strategy Parameters
N = 60                      # Bars for accumulation phase
threshold = 0.01            # Volatility threshold (1%)
total_capital = 100000     # Total capital (₹10 lakh)
risk_per_trade = 0.008       # Fixed risk per trade (3% of capital)
slippage_rate = 0.0001      # 0.01% slippage
brokerage_rate = 0.0005      # 0.1% brokerage
risk_reward_ratio = 3     # Risk-Reward Ratio (3:1)

# Data Preprocessing
data['date'] = pd.to_datetime(data['date'])
data = data.sort_values(by=['Ticker', 'date'])

# Precompute Indicators
data['prev_max_high'] = data.groupby('Ticker')['high'].transform(lambda x: x.shift(1).rolling(N-1).max())
data['prev_min_low'] = data.groupby('Ticker')['low'].transform(lambda x: x.shift(1).rolling(N-1).min())
data['std_close'] = data.groupby('Ticker')['close'].transform(lambda x: x.rolling(N).std())
data['avg_volume'] = data.groupby('Ticker')['volume'].transform(lambda x: x.rolling(N).mean())
data['VWAP_Deviation'] = (data['close'] - data['VWAP']) / data['VWAP']

tickers = data['Ticker'].unique()

def process_ticker(ticker):
    """Process a single ticker and apply the trading strategy."""
    ticker_data = data[data['Ticker'] == ticker].copy()
    ticker_data['day'] = ticker_data['date'].dt.date
    days = ticker_data['day'].unique()
    tradebook = []
    cumulative_pnl = 0

    for day in days:
        day_data = ticker_data[ticker_data['day'] == day].copy()

        # Convert to NumPy arrays for speed
        close_prices = day_data['close'].values
        high_prices = day_data['high'].values
        low_prices = day_data['low'].values
        volume = day_data['volume'].values
        vwap_deviation = day_data['VWAP_Deviation'].values
        std_close = day_data['std_close'].values
        prev_max_high = day_data['prev_max_high'].values
        prev_min_low = day_data['prev_min_low'].values
        avg_volume = day_data['avg_volume'].values
        adx = day_data['ADX'].values
        mfi = day_data['MFI'].values
        atr = day_data['ATR'].values  # ATR for TP calculation

        # Trade timing
        trade_cutoff_time = pd.Timestamp(f"{day} 15:15:00")  # Last entry
        exit_time = pd.Timestamp(f"{day} 15:25:00")          # Force exit

        in_trade = False
        trade_type = None
        entry_price = entry_time = shares = stop_loss = take_profit = 0

        for i in range(N, len(day_data)):
            current_time = day_data['date'].iloc[i]

            # Exit trade if SL, TP, or EOD is hit
            if in_trade:
                if trade_type == 'BUY':
                    if low_prices[i] <= stop_loss:
                        exit_price = stop_loss
                        exit_reason = 'Stop-Loss Hit'
                    elif high_prices[i] >= take_profit:
                        exit_price = take_profit
                        exit_reason = 'Take-Profit Hit'
                    elif current_time >= exit_time:
                        exit_price = close_prices[i]
                        exit_reason = 'End of Day'
                    else:
                        continue
                elif trade_type == 'SELL':
                    if high_prices[i] >= stop_loss:
                        exit_price = stop_loss
                        exit_reason = 'Stop-Loss Hit'
                    elif low_prices[i] <= take_profit:
                        exit_price = take_profit
                        exit_reason = 'Take-Profit Hit'
                    elif current_time >= exit_time:
                        exit_price = close_prices[i]
                        exit_reason = 'End of Day'
                    else:
                        continue

                # Adjust for slippage
                effective_entry = entry_price * (1 + slippage_rate if trade_type == 'BUY' else 1 - slippage_rate)
                effective_exit = exit_price * (1 - slippage_rate if trade_type == 'BUY' else 1 + slippage_rate)

                # Calculate Profit & Loss
                gross_profit = (effective_exit - effective_entry) * shares * (1 if trade_type == 'BUY' else -1)
                brokerage = brokerage_rate * (entry_price * shares + exit_price * shares)
                net_profit = gross_profit - brokerage
                cumulative_pnl += net_profit

                # Log the trade
                tradebook.append({
                    'Ticker': ticker, 'Trade Type': trade_type, 'Entry Time': entry_time, 'Entry Price': entry_price,
                    'Exit Time': current_time, 'Exit Price': exit_price, 'Shares': shares,
                    'PnL': net_profit, 'Cumulative PnL': cumulative_pnl, 'Exit Reason': exit_reason
                })
                in_trade = False

            # Detect accumulation phase (50% of N bars meet conditions)
            if not in_trade and current_time < trade_cutoff_time:
                accumulation_score = 0
                for j in range(i - N + 1, i + 1):
                    if std_close[j] < threshold * close_prices[j] and \
                       volume[j] > 1.2 * avg_volume[j] and \
                       abs(vwap_deviation[j]) < 0.01:
                        accumulation_score += 1
                if accumulation_score >= 0.5 * N:
                    risk_amount = risk_per_trade * total_capital  # Fixed ₹30,000 risk per trade

                    # BUY setup
                    if high_prices[i] > prev_max_high[i] and adx[i] > 20 and mfi[i] > 60 and volume[i] > 1.25 * avg_volume[i]:
                        trade_type = 'BUY'
                        entry_price = close_prices[i]
                        stop_loss = entry_price - (risk_amount / total_capital) * entry_price  # Fixed 3% SL
                        take_profit = entry_price + (risk_reward_ratio * atr[i])  # ATR-based TP
                        shares = int(risk_amount / (entry_price - stop_loss))
                        in_trade = True
                        entry_time = current_time

                    # SELL setup
                    elif low_prices[i] < prev_min_low[i] and adx[i] > 20 and mfi[i] < 40 and volume[i] > 1.25 * avg_volume[i]:
                        trade_type = 'SELL'
                        entry_price = close_prices[i]
                        stop_loss = entry_price + (risk_amount / total_capital) * entry_price  # Fixed 3% SL
                        take_profit = entry_price - (risk_reward_ratio * atr[i])  # ATR-based TP
                        shares = int(risk_amount / (stop_loss - entry_price))
                        in_trade = True
                        entry_time = current_time

    return tradebook

# Run strategy in parallel for all tickers
results = Parallel(n_jobs=-1)(delayed(process_ticker)(ticker) for ticker in tqdm(tickers, desc="Processing Tickers"))

# Convert to DataFrame
tradebook_df = pd.DataFrame([trade for result in results for trade in result])

# Display results
print(tradebook_df)


Processing Tickers: 100%|██████████| 48/48 [26:43<00:00, 33.40s/it]


         Ticker Trade Type          Entry Time  Entry Price  \
0      ADANIENT       SELL 2015-02-05 14:14:00       636.50   
1      ADANIENT       SELL 2015-02-05 14:16:00       631.00   
2      ADANIENT        BUY 2015-02-11 11:00:00       647.70   
3      ADANIENT        BUY 2015-02-11 11:06:00       655.00   
4      ADANIENT       SELL 2015-02-13 15:12:00       672.45   
...         ...        ...                 ...          ...   
23414     WIPRO       SELL 2024-09-11 14:30:00       516.60   
23415     WIPRO        BUY 2024-09-23 14:29:00       534.65   
23416     WIPRO        BUY 2024-09-26 15:00:00       542.15   
23417     WIPRO       SELL 2024-10-18 15:14:00       547.95   
23418     WIPRO        BUY 2024-11-04 15:13:00       540.85   

                Exit Time  Exit Price  Shares         PnL  Cumulative PnL  \
0     2015-02-05 14:16:00  631.635714     157  644.234473      644.234473   
1     2015-02-05 14:42:00  626.028571     158  666.319406     1310.553879   
2     2015-0

In [ ]:
# Optional: Save to CSV
tradebook_df.to_csv('/content/drive/MyDrive/tradebook.csv', index=False)

In [ ]:
tradebook_df

,Ticker,Trade Type,Entry Time,Entry Price,Exit Time,Exit Price,Shares,PnL,Cumulative PnL,Exit Reason
0,ADANIENT,SELL,2015-02-05 14:14:00,636.50,2015-02-05 14:16:00,631.635714,157,644.234473,644.234473,Take-Profit Hit
1,ADANIENT,SELL,2015-02-05 14:16:00,631.00,2015-02-05 14:42:00,626.028571,158,666.319406,1310.553879,Take-Profit Hit
2,ADANIENT,BUY,2015-02-11 11:00:00,647.70,2015-02-11 11:06:00,653.517857,154,775.717470,2086.271349,Take-Profit Hit
3,ADANIENT,BUY,2015-02-11 11:06:00,655.00,2015-02-11 12:34:00,661.203571,152,822.905091,2909.176440,Take-Profit Hit
4,ADANIENT,SELL,2015-02-13 15:12:00,672.45,2015-02-13 15:17:00,668.967857,148,396.239237,3305.415677,Take-Profit Hit
...,...,...,...,...,...,...,...,...,...,...
23414,WIPRO,SELL,2024-09-11 14:30:00,516.60,2024-09-11 14:51:00,514.532143,193,279.691326,-14931.976381,Take-Profit Hit
23415,WIPRO,BUY,2024-09-23 14:29:00,534.65,2024-09-23 15:00:00,535.839286,187,102.287531,-14829.688850,Take-Profit Hit
23416,WIPRO,BUY,2024-09-26 15:00:00,542.15,2024-09-26 15:25:00,542.350000,184,-82.928800,-14912.617650,End of Day
23417,WIPRO,SELL,2024-10-18 15:14:00,547.95,2024-10-18 15:25:00,547.900000,182,-110.566820,-15023.184470,End of Day
